In [ ]:
from Binoculars.binoculars.detector import Binoculars

In [ ]:
from tqdm import tqdm_notebook as tqdm
bino = Binoculars(
    observer_name_or_path='Qwen/Qwen2.5-7B',
    performer_name_or_path='Qwen/Qwen2.5-7B-Instruct'
)

In [ ]:
# ChatGPT (GPT-4) output when prompted with “Can you write a few sentences about a capybara that is an astrophysicist?"
sample_string = '''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
groundbreaking research in astrophysics.'''

print(bino.compute_score(sample_string))  # 0.75661373
print(bino.predict(sample_string))  # 'Most likely AI-Generated'

In [ ]:
import pandas as pd
multisocial = pd.read_csv("../../../data/external_data/multisocial/multisocial_anonymized.csv")
multisocial = multisocial[multisocial.split == "test"]
print(len(multisocial))

In [ ]:
from tqdm import tqdm
import torch
import gc

batch_size = 16

scores = []
for start in tqdm(range(0, len(multisocial), batch_size), desc="Progress"):
    end = start + batch_size
    batch_text = multisocial["text"].iloc[start:end].tolist()
    scores.extend(bino.compute_score(batch_text))

    # Clean cache:
    del batch_text
    torch.cuda.empty_cache()
    gc.collect()

    if len(scores) % 10000 == 0:
      data_sample = multisocial.iloc[:len(scores)]
      data_sample["bino_score"] = scores
      data_sample.to_parquet("multisocial_bino_scores.parquet")

multisocial["bino_score"] = scores
multisocial.to_parquet("multisocial_bino_scores.parquet")

In [ ]:
import pandas as pd
AIGTBench = pd.read_parquet("../../../data/external_data/AIGTBench")
AIGTBench = AIGTBench[AIGTBench.social_media_platform == "reddit"]
print(len(AIGTBench))

In [ ]:
AIGTBench_calculated = pd.read_parquet("AIGTBench_bino_scores.parquet")

In [ ]:
from tqdm import tqdm
import torch
import gc
# scores = []
batch_size = 16
scores = AIGTBench_calculated.bino_score.to_list()
for start in tqdm(range(len(scores), len(AIGTBench), batch_size), desc="Progress"):
    end = start + batch_size
    batch_text = AIGTBench["text"].iloc[start:end].tolist()
    scores.extend(bino.compute_score(batch_text))

    # Clean cache:
    del batch_text
    torch.cuda.empty_cache()
    gc.collect()

    if len(scores) % 10000 == 0:
      data_sample = AIGTBench.iloc[:len(scores)]
      data_sample["bino_score"] = scores
      data_sample.to_parquet("AIGTBench_bino_scores.parquet")

AIGTBench["bino_score"] = scores
AIGTBench.to_parquet("AIGTBench_bino_scores.parquet")